In [5]:
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import pprint
from openai import OpenAI

# 1. Risoluzione del percorso per importare i moduli dalla cartella 'src'
sys.path.append(os.path.abspath(os.path.join("..", "src")))

from auditor_core import scarica_ultimo_log, valuta_chat_con_gemma, salva_valutazione_db

# --- CONFIGURAZIONE E SETUP ---
API_KEY_HERO = os.getenv("PLATFORMHERO_TOKEN")
URL_API_HERO = os.getenv("PLATFORMHERO_URL")

# Configurazione del modello locale performante (Llama 3 8B)
MODEL_NAME = "llama3"
client_openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama-local")

# Percorso relativo corretto partendo dalla cartella 'notebooks/'
DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")

def mia_connessione():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

print("Sistema pronto e funzioni caricate!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Sistema pronto e funzioni caricate!


In [ ]:
#  Scarica il log reale di produzione
log_reale = scarica_ultimo_log(API_KEY_HERO, URL_API_HERO)

# 🎯 ESTRAZIONE CORRETTA: Scaviamo dentro i veri campi del dizionario
testo_ticket = log_reale.get("content", {}).get("message", {}).get("content", "")
id_del_log = log_reale.get("log_id", "ID_Mancante")

print(" Log agganciato correttamente")
print(f" ID Log: {id_del_log}")
print(f" Caratteri estratti per la valutazione: {len(testo_ticket)}")

# Valuta con Gemma (Ora riceverà il testo vero)
risposta_ai = valuta_chat_con_gemma(client_openai, MODEL_NAME, testo_ticket)

# Salva sul database SQLite usando il vero 'log_id'
salva_valutazione_db(mia_connessione, id_del_log, risposta_ai)

 Log agganciato correttamente!
 ID Log: QLmh8L9QyGIVGiVHNh8V7NWP
 Caratteri estratti per la valutazione: 661
 Successo! Valutazione registrata. Voto LLM: 10.0/10.0


In [3]:
print(" ECCO COSA HA REALIENMTE SCRITTO GEMMA:")
print(risposta_ai)

 ECCO COSA HA REALIENMTE SCRITTO GEMMA:
{"system_judicial_reasoning": "L'agente identifica correttamente la causa tecnica del timeout (mancanza di filtri) ed elabora una soluzione pratica e strutturata fornendo all'utente opzioni chiare per procedere.", "technical_score": 10.0, "completeness_score": 9.5, "completeness_proactivity": 10.0, "business_score": 10.0, "consistency_score": 10.0, "overall_score": 10.0, "feedback": "Eccellente gestione del problema tecnico con chiare opzioni di risoluzione.", "issues": ""}


In [4]:
# Vediamo quali chiavi esistono davvero in questo log di produzione
print(" CHIAVI REALI DEL LOG:", list(log_reale.keys()))

# Estraiamo il contenuto inviato a Gemma per vederne la lunghezza
contenuto_chat = log_reale.get("chat_history", "NON TROVATO")
print(f"\n LUNGHEZZA DEL TESTO CHAT: {len(str(contenuto_chat))} caratteri")

print("\n ANTEPRIMA DEI DATI COMPLETI DEL LOG:")
pprint.pprint(dict(log_reale))

 CHIAVI REALI DEL LOG: ['object', 'log_id', 'organization_id', 'log_type', 'model_id', 'collection_id', 'assistant_id', 'agent_id', 'admin_id', 'apikey_id', 'chat_id', 'message_id', 'session_id', 'event', 'event_id', 'event_step', 'tool_type', 'tool_id', 'input_tokens', 'output_tokens', 'total_tokens', 'input_token_cost', 'output_token_cost', 'total_cost', 'cost_currency', 'content', 'event_timestamp', 'created_timestamp', 'updated_timestamp']

 LUNGHEZZA DEL TESTO CHAT: 11 caratteri

 ANTEPRIMA DEI DATI COMPLETI DEL LOG:
{'admin_id': 'X3ArdAqXkFjN4qbN',
 'agent_id': None,
 'apikey_id': None,
 'assistant_id': 'X5lMJbD4B2EOwpwahSkW4TnW',
 'chat_id': 'SdELmOztk69ymKWDO20MloQ4',
 'collection_id': None,
 'content': {'message': {'content': 'La lettura di **SprintEmployees** è '
                                    'andata in timeout (oltre 100s) perché '
                                    'l’estrazione “tutta la tabella” '
                                    '(`top=10000` senza filtri) è tr